In [1]:
# Capstone project work
# Author Padmanabhan S Pillai, Dec 1 2025
# Notebook works towards building the project, based on the course material
# Project would be deployed to Vertex AgentEngine
# This is not to a guide to building a production quality code but to work through concepts around buiding agents, in the current form.

# Modules required for Code development and deployment


In [2]:
import os
import random
import time
import vertexai
from kaggle_secrets import UserSecretsClient
from vertexai import agent_engines

print("✅ Development and Deployment module imports completed successfully")

✅ Development and Deployment module imports completed successfully


# Attach GCP account

using menu "Add-ons-->Google Cloud SDK" and completing the Oauth flow, Code below sets the temporary credentials to work with GCP services using GCP SDK

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
user_credential = user_secrets.get_gcloud_credential()
user_secrets.set_tensorflow_credential(user_credential)

print("✅ GCP Cloud credentials configured")

✅ GCP Cloud credentials configured


# Set the GCP project

**Enable API for Vertex deployment**

It is important that for deploying successfully the agent to the Vertex AgentEngine, the following API & Services are enabled in the GCP project.
- Vertex AI API
- Cloud Storage API
- Cloud Logging API
- Cloud Monitoring API
- Cloud Trace API
- Telemetry API.
  
**Enable API for Map MCP service**

To work with maps MCP tools from google ,  https://mapstools.googleapis.com/mcp additional services listes below are enabled and the API key restrictioon list includes the below AP/Services**
- Maps Grounding Lite API
- Directions API
- Geocoding API
- Places API *( Please note not the Places API(New))*
- Routes API and
- Weather API

**Enable MCP service access for Maps Grounding Lite API**

Additionally it is important to make sure MCP service is enabled in the GCP project, for that execute the below command using gcloud client tool.**

```bash
gcloud beta services mcp enable mapstools.googleapis.com --project=<your-project-number>
```
*Please note that the project number is used in the above call, not the id*

# Set API key, GCP PROJECT_ID and GIT repo as OS environment variable

1. Using the menu "Add-ons--> Secrets" store the API key associated with the project in the kaggle secret manager and enable it by checking the box
2. Execute the following code to set the environment variable
3. Please note that code executing from Vertex AgentMachine does not require tke API key to work with LLM, at the same time the MCP service do need this.


In [4]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    GOOGLE_CLOUD_PROJECT = UserSecretsClient().get_secret("GOOGLE_CLOUD_PROJECT")
    os.environ["GOOGLE_CLOUD_PROJECT"] = GOOGLE_CLOUD_PROJECT
    print("✅ Setup environment variables with API Key and Project Id complete.")
    os.environ["GOOGLE_CLOUD_PROJECT"] = GOOGLE_CLOUD_PROJECT
    print("✅ Setup environment variables with API Key and Project Id complete.")
    os.environ["GITREPO"] = GITREPO
    print("✅ Setup environment variables with API Key and Project Id complete.")
except Exception as e:
    print(
        f"Error setting APIKEY and PROJECT ID from  Kaggle secrets to env variable. Details: {e}"
    )

✅ Setup environment variables with API Key and Project Id complete.


# Concierge Agent

Implemented using Multi Agent Architecture

---

<img width="800" src="https://github.com/pnabhans/kaggle5dayAgentCaptstone/blob/init_feature/img/AgentArchitecture.png?raw=true" alt="Concierge Agent" />

---



## Expected skills/Objectives
   Guest (aka user) interacts with the **Guest Manager** Agent
   
        - discuss about a new assistance required or updates about their decision about an earlier assistance requested awaiting a decision
        
   ### Guest Manager

   
  - Mark uniquely a new assistance request so that it can be addressed across session and for a period of time   
  - Based on the kind of assistance requested, will delegate the the request to ***Travel***, ***Advisory***, or ***Accomodation*** agent
  - If the assistance have been previously requested and awating a response from Guest the decision is forwarded to the ***Closure*** 
  agent bypasssing the other agents as a closure is what is required

        
   ### Travel Agent
   
   - Every unique request for travel assistance is handled by this Agent
   - Will prepare travel plans and prepare the details of travel like mode of travel, details like distance, time of travel
   - Can identify places by mentions as in normal language like Los angeles civic center, Consular services of country etc
   - Will share its findings to the ***Closure*** Agent
   - it is constituted of 2 sub agents, one to identify places and do routing, ***Routing*** agent and ***Flight*** agent which adds real flight information to the input provided by *Routing* agent, if required, and produce a complete relevant travel plan
   - The ***Routing*** agent use tools provided by **MCP**, ***maps-grounding-lite-mcp*** , which is MCP service over google maps api services to gain skill required
   - The **MCP** toolset is pruned to get only the required tools ( *search_places* and *compute_route*) to manage the ***Context** size
   - The ***Flight*** agent use ***google_search*** inbuilt tool
        
   ### Advisory Agent
   
   - Based on the request if there are travel, weather or otherwise advisories are present it adds to information provided to guest
   - Pass the information to ***Closure*** Agent to take futher action
   - Uses inbuilt tool ***google_search*** 
        
   ### Accomodation Agent
   
   - Any outside stay related information need to be gathered and made available this agent will work on that
   - Will provide its findings to ***Closure*** Agent for further action
   - Uses inbuilt tool  ***google_search***

   ### Closure Agent
   
   - For every uniquely identified requested assistance this agent will gather from ***memory*** history to identify the current status and either wait for a completion in terms of all parts being available ( like travel, advisory, accomodation) or an ***Human in the loop*** approval/denial of service
   - Due to the basic nature of this process, a particular request can be processed across multiple sessions, days or months
   - Once closure is achieved will provide input to ***Fulfillment*** agent to get *guest*/*user* the final fulfillments
   - Uses ***function based*** tool

  ### Fulfillment Agent
  
  - Based on the input provided by ***Closure*** agent provide intermediate or final fullfillment
  - Completes an unique assistance request
    


---
# AgentEngine Deploy Setup
Following folder structure is built to enable deployment to Vertex AgentEngine 

<small>Please note that for the repository reasons. This folder structure is build under the local repository clone , root</small>

---

```
concierge/
├── agent.py                  # The logic
├── requirements.txt          # The libraries
├── .env                      # The secrets/config
└── .agent_engine_config.json # The hardware specs
```

In [5]:
# If the deploy folder was not setup on prior execution set it up
import os
from pathlib import Path

# 1. Define your configuration Variables
# The standard writable directory in Kaggle
KAGGLE_WORKING_DIR = "/kaggle/working"
# The specific folder name your git repo will create
REPO_FOLDER_NAME = "kaggle5dayAgentCaptstone"
# Construct the full path to where the repo should be
FULL_REPO_PATH = os.path.join(KAGGLE_WORKING_DIR, REPO_FOLDER_NAME)

# Get the git URL from environment variables (ensure this is set previously!)
GIT_REPO_URL = os.environ.get("GITREPO")

if not GIT_REPO_URL:
    print("❌ Error: GITREPO environment variable is not set.")
else:
    # 2. Ensure we are starting in the base working directory
    print(f"Moving to base directory: {KAGGLE_WORKING_DIR}")
    %cd {KAGGLE_WORKING_DIR}

    # 3. The Core Logic Check
    if os.path.exists(FULL_REPO_PATH):
        # --- Scenario A: The folder already exists ---
        print(f"✅ Repository folder found at: {FULL_REPO_PATH}")
        print("Switching into repository directory...")
        # Use magic %cd so the change sticks for subsequent cells
        %cd {FULL_REPO_PATH}
        
        # Optional: It's often good practice to ensure it's up to date
        print("Attempting to pull latest changes...")
        !git pull

    else:
        # --- Scenario B: The folder does not exist ---
        print(f"Folder not found. Cloning repository from: {GIT_REPO_URL}")
        # Run the clone command
        !git clone {GIT_REPO_URL}
        
        # Verify clone was successful before switching
        if os.path.exists(FULL_REPO_PATH):
             print("Clone complete. Switching into repository directory...")
             %cd {FULL_REPO_PATH}
        else:
             print("❌ Error: Git clone failed.")

# 4. Final check to see where we ended up
print("-" * 20)
print("Current working directory:")
!pwd

Cloning into 'kaggle5dayAgentCaptstone'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 52 (delta 17), reused 11 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 10.63 MiB | 43.21 MiB/s, done.
Resolving deltas: 100% (17/17), done.


In [2]:
%cd kaggle5dayAgentCaptstone

[Errno 2] No such file or directory: 'kaggle5dayAgentCaptstone'
/kaggle/working


In [10]:
# Change to feature branch
!git status
!git switch init_feature
!git status
!ls


On branch init_feature
Your branch is up to date with 'origin/init_feature'.

nothing to commit, working tree clean
Already on 'init_feature'
Your branch is up to date with 'origin/init_feature'.
On branch init_feature
Your branch is up to date with 'origin/init_feature'.

nothing to commit, working tree clean
img  notebooks	README.md


**Add .gitignore**

In [14]:
%%writefile ./.gitignore
# Byte-compiled / optimized / DLL files
__pycache__/
*.py[cod]
*$py.class

# C extensions
*.so

# Distribution / packaging
.Python
build/
develop-eggs/
dist/
downloads/
eggs/
.eggs/
lib/
lib64/
parts/
sdist/
var/
wheels/
share/python-wheels/
*.egg-info/
.installed.cfg
*.egg
MANIFEST

# Virtual Environments
.env
.venv
env/
venv/
ENV/

# IDEs
.vscode/
.idea/

# ADK specific
.adk/
adk_deploy/


Writing ./.gitignore


In [15]:
!cat .gitignore

# Byte-compiled / optimized / DLL files
__pycache__/
*.py[cod]
*$py.class

# C extensions
*.so

# Distribution / packaging
.Python
build/
develop-eggs/
dist/
downloads/
eggs/
.eggs/
lib/
lib64/
parts/
sdist/
var/
wheels/
share/python-wheels/
*.egg-info/
.installed.cfg
*.egg
MANIFEST

# Virtual Environments
.env
.venv
env/
venv/
ENV/

# IDEs
.vscode/
.idea/

# ADK specific
.adk/
adk_deploy/


**Create the required folder structure**


/kaggle/working


In [ ]:
!mkdir concierge